## 🛠️ Configuración del Entorno y Librerías

In [2]:
import pandas as pd 
import numpy as np

## 📂 Carga de los Datasets Principales

In [3]:
df_main = pd.read_csv('../data/diabetic_data.csv')
df_ids = pd.read_csv('../data/ids_mapping.csv')

## 🔍 Inspección Inicial de los Datos

In [4]:
# Visualizamos los primeros registros del dataframe principal
print("------- Primeros REgistros de daibetic_data.csv -------")
display(df_main.head())

# Visualizamos los primeros registros del dataframe de mapeo de IDs
print("------- Primeros Registros de ids_mapping.csv -------")
display(df_ids.head())

------- Primeros REgistros de daibetic_data.csv -------


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


------- Primeros Registros de ids_mapping.csv -------


,admission_type_id,description
0,1,Emergency
1,2,Urgent
2,3,Elective
3,4,Newborn
4,5,Not Available


## 🛠️ Estandarización de Valores 

In [5]:
# Convertimos todos los valores faltantes ? a NaN (Not a NUmber)
df_main.replace('?', np.nan, inplace=True)

## 📊 Resumen de Estructura y Calidad de Datos

In [6]:
# Creo un resumen de los datos para entender su estructura y contenido
print("------- Resumen del dataframe principal -------")    

resumen = pd.DataFrame({
    'Columnas': df_main.columns,
    'Tipos': df_main.dtypes,
    'No nulos': df_main.notnull().sum(),
    'Valores Nulos': df_main.isnull().sum()
})


print("------- Resumen del dataframe ids -------")
resumen_ids = pd.DataFrame({
    'Columnas': df_ids.columns,
    'Tipos': df_ids.dtypes,
    'No nulos': df_ids.notnull().sum(),
    'Valores Nulos': df_ids.isnull().sum()
})

# FUerzo a que me muestre todas las filas del resumen creado

with pd.option_context('display.max_rows', None):
    display(resumen)
    display(resumen_ids)

------- Resumen del dataframe principal -------
------- Resumen del dataframe ids -------


,Columnas,Tipos,No nulos,Valores Nulos
encounter_id,encounter_id,int64,101766,0
patient_nbr,patient_nbr,int64,101766,0
race,race,object,99493,2273
gender,gender,object,101766,0
age,age,object,101766,0
weight,weight,object,3197,98569
admission_type_id,admission_type_id,int64,101766,0
discharge_disposition_id,discharge_disposition_id,int64,101766,0
admission_source_id,admission_source_id,int64,101766,0
time_in_hospital,time_in_hospital,int64,101766,0


,Columnas,Tipos,No nulos,Valores Nulos
admission_type_id,admission_type_id,object,65,2
description,description,object,62,5


## 🛠️ Mejora de variables categoricas

In [7]:
# Definimos las columnas que vamos a modificar

cols_to_fix = ['race', 'gender', 'payer_code', 'medical_specialty']

# Agregamos todas las variantes que suelen aparecer en este dataset
df_main[cols_to_fix] = df_main[cols_to_fix].replace(['Unknown/Invalid'], 'Other')

#Remplazamos los valores nulos por Other para poder trabajar con ellos posteriormente
df_main[cols_to_fix] = df_main[cols_to_fix].fillna('Other')

# Verificamos que ya no haya valores nulos en las columnas seleccionadas
print("------- Verificación de valores nulos en columnas seleccionadas -------")    
print(df_main[cols_to_fix].isnull().sum())

------- Verificación de valores nulos en columnas seleccionadas -------
race                 0
gender               0
payer_code           0
medical_specialty    0
dtype: int64


## 📏 Dimensiones de los Conjuntos de Datos

In [8]:
# Verificamos la dimension del conjunto de datos de los dataframes
print("------- Dimensión del dataframe principal -------")
print(df_main.shape)
print("------- Dimensión del dataframe ids -------")
print(df_ids.shape)

------- Dimensión del dataframe principal -------
(101766, 50)
------- Dimensión del dataframe ids -------
(67, 2)


## 🗑️ Optimización de la Estructura por Integridad de Datos

In [9]:
# ELiminamos las columnas que no vamos a utilizar para el análisis por cantidad de valores nulos mas del 90%
cols_to_drop = ['weight', 'A1Cresult', 'max_glu_serum']
df_main.drop(columns=cols_to_drop, inplace=True)
print("------- Columnas eliminadas por tener más del 90% de valores nulos -------")
print(cols_to_drop) 

------- Columnas eliminadas por tener más del 90% de valores nulos -------
['weight', 'A1Cresult', 'max_glu_serum']


In [10]:
# Verificamos que eliminamos las columnas correctamente del dataframe principal
print("------- Columnas restantes en el dataframe principal -------")   
print(df_main.shape)

------- Columnas restantes en el dataframe principal -------
(101766, 47)


## 🩺 Estandarización y Agrupación de Diagnósticos (ICD-9)

In [11]:
# --- 2. ESTANDARIZACIÓN DE DIAGNÓSTICOS (Agrupación ICD-9) ---
def map_cie9(code):
    try:
        if 'V' in str(code) or 'E' in str(code): return 'Other'
        code = float(code)
        if 390 <= code <= 459 or code == 785: return 'Circulatory'
        elif 460 <= code <= 519 or code == 786: return 'Respiratory'
        elif 520 <= code <= 579 or code == 787: return 'Digestive'
        elif int(code) == 250: return 'Diabetes'
        elif 800 <= code <= 999: return 'Injury'
        elif 710 <= code <= 739: return 'Musculoskeletal'
        elif 580 <= code <= 629 or code == 788: return 'Genitourinary'
        else: return 'Neoplasms/Other'
    except:
        return 'Other'

for col in ['diag_1', 'diag_2', 'diag_3']:
    df_main[f'{col}_group'] = df_main[col].apply(map_cie9)



In [12]:
# Verificamos que eliminamos las columnas correctamente del dataframe principal
print("------- Columnas restantes en el dataframe principal -------")   
print(df_main.shape)

------- Columnas restantes en el dataframe principal -------
(101766, 50)


## 🩺 Estandarización y Agrupación de Medicamentos

In [ ]:

# creamos la lista de medicamentos que vamos a analizar para contar los cambios en el tratamiento
medications = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride',
    'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone',
    'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide',
    'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin',
    'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone'
]

# 1. Primero calculamos el número de cambios en el tratamiento para cada paciente
df_main['med_change_count'] = df_main[medications].apply(lambda x: x.isin(['Up', 'Down']).sum(), axis=1)

# 2. Definimos las etiquetas según el número de cambios
def label_stability(count):
    if count == 0:
        return "Stable"
    elif count == 1:
        return "Low Instability"
    else:
        return "High Instability"

# 3. Creamos la nueva columna con palabras
df_main['treatment_status'] = df_main['med_change_count'].apply(label_stability)



print(df_main['treatment_status'].value_counts())

treatment_status
Stable              74063
Low Instability     26272
High Instability     1431
Name: count, dtype: int64


## 📊 Estandarización de la variable Reingreso 

In [ ]:
# 1. Definimos la función para traducir los tres casos de reingreso
def labels_reingreso(valor):
    if valor == '<30':
        return "Reingreso Temprano (<30 días)"
    elif valor == '>30':
        return "Reingreso Tardío (>30 días)"
    else:
        return "No Reingresó"

# 2. Creamos la nueva columna con las palabras descriptivas
df_main['readmitted_status'] = df_main['readmitted'].apply(labels_reingreso)

# Verificamos cómo quedaron distribuidos
print(df_main['readmitted_status'].value_counts())

readmitted_status
No Reingresó                     54864
Reingreso Tardío (>30 días)      35545
Reingreso Temprano (<30 días)    11357
Name: count, dtype: int64


In [15]:
# Borramos la columna original 'readmitted' y nos quedamos con la ya traducida 'readmitted_status'
df_main.drop(columns=['readmitted'], inplace=True)

# Verificamos que se haya ido y qué columnas nos quedan
print("Columnas actuales en el dataset:")
print(df_main.columns)

Columnas actuales en el dataset:
Index(['encounter_id', 'patient_nbr', 'race', 'gender', 'age',
       'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
       'time_in_hospital', 'payer_code', 'medical_specialty',
       'num_lab_procedures', 'num_procedures', 'num_medications',
       'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1',
       'diag_2', 'diag_3', 'number_diagnoses', 'metformin', 'repaglinide',
       'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide',
       'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone',
       'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide',
       'examide', 'citoglipton', 'insulin', 'glyburide-metformin',
       'glipizide-metformin', 'glimepiride-pioglitazone',
       'metformin-rosiglitazone', 'metformin-pioglitazone', 'change',
       'diabetesMed', 'diag_1_group', 'diag_2_group', 'diag_3_group',
       'med_change_count', 'treatment_status', 'readmitted_status

## 🔢 Estandarizacion de Rangos Etarios

In [16]:
# Estandarzamos la columna de edad convirtiendo los rangos a un valor numérico representativo (el punto medio del rango)
age_map = {
    '[0-10)': 5, '[10-20)': 15, '[20-30)': 25, '[30-40)': 35, 
    '[40-50)': 45, '[50-60)': 55, '[60-70)': 65, '[70-80)': 75, 
    '[80-90)': 85, '[90-100)': 95
}
df_main['age_numeric'] = df_main['age'].map(age_map)

In [17]:
# Eliminamos la columna de texto original y mantenemosos la nueva columna numérica age_numeric
df_main.drop(columns=['age'], inplace=True)

# Verificamos que ahora solo tenemos la versión numérica
print(df_main[['age_numeric']].head())

   age_numeric
0            5
1           15
2           25
3           35
4           45


## 🏥 Segmentación de la Estancia Hospitalaria

In [18]:
# 1. Creamos la función que categoriza según el número de días de hospitalización
def categorize_stay_length(days):
    if days == 1:
        return "Short/Ambulatory"
    elif 2 <= days <= 4:
        return "Standard"
    elif 5 <= days <= 8:
        return "Moderate"
    elif days >= 9:
        return "Prolonged"
    else:
        return "Unknown"

# 2. Aplicamos la función y creamos la nueva columna
df_main['hospital_stay_type'] = df_main['time_in_hospital'].apply(categorize_stay_length)

# 3. Verificamos los resultados
print("--- Distribution of Hospital Stay Types ---")
print(df_main['hospital_stay_type'].value_counts())

--- Distribution of Hospital Stay Types ---
hospital_stay_type
Standard            48904
Moderate            27755
Short/Ambulatory    14208
Prolonged           10899
Name: count, dtype: int64


## 🧹 Eliminación de variagles originales de Diagnostico

In [19]:
# Lista de columnas originales que ya estandarizamos
cols_to_remove = ['diag_1', 'diag_2', 'diag_3']

# Las eliminamos del DataFrame
df_main.drop(columns=cols_to_remove, inplace=True)

print("✅ Columnas de diagnósticos originales eliminadas. El dataset está más limpio.")

✅ Columnas de diagnósticos originales eliminadas. El dataset está más limpio.


In [20]:
# Verificamos las columnas del dataset para asegurarnos de que se eliminaron las columnas originales de diagnóstico
print("------- Columnas actuales en el dataset después de eliminar diagnósticos originales -------")
print(df_main.columns)

------- Columnas actuales en el dataset después de eliminar diagnósticos originales -------
Index(['encounter_id', 'patient_nbr', 'race', 'gender', 'admission_type_id',
       'discharge_disposition_id', 'admission_source_id', 'time_in_hospital',
       'payer_code', 'medical_specialty', 'num_lab_procedures',
       'num_procedures', 'num_medications', 'number_outpatient',
       'number_emergency', 'number_inpatient', 'number_diagnoses', 'metformin',
       'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride',
       'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
       'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
       'tolazamide', 'examide', 'citoglipton', 'insulin',
       'glyburide-metformin', 'glipizide-metformin',
       'glimepiride-pioglitazone', 'metformin-rosiglitazone',
       'metformin-pioglitazone', 'change', 'diabetesMed', 'diag_1_group',
       'diag_2_group', 'diag_3_group', 'med_change_count', 'treatment_status',
   

## 🧹 Limpieza y Normalización del Diccionario de Mapeo

In [21]:
mapping = df_ids.copy()

# Crear columna category
mapping["category"] = None

current_category = "admission_type"

for i in range(len(mapping)):
    val = str(mapping.loc[i, "admission_type_id"])
    
    if "discharge_disposition_id" in val:
        current_category = "discharge_disposition"
    elif "admission_source_id" in val:
        current_category = "admission_source"
    
    mapping.loc[i, "category"] = current_category

# Eliminar filas header
mapping_clean = mapping[
    ~mapping["admission_type_id"].isin([
        "admission_type_id",
        "discharge_disposition_id",
        "admission_source_id"
    ])
].copy()

# Renombrar
mapping_clean.columns = ["id", "description", "category"]

# Convertir ID
mapping_clean["id"] = pd.to_numeric(mapping_clean["id"], errors="coerce")

mapping_clean.head()

,id,description,category
0,1.0,Emergency,admission_type
1,2.0,Urgent,admission_type
2,3.0,Elective,admission_type
3,4.0,Newborn,admission_type
4,5.0,Not Available,admission_type


## 📂 Fragmentación del Diccionario en Tablas

In [22]:
admission_map = mapping_clean[mapping_clean["category"] == "admission_type"].copy()
discharge_map = mapping_clean[mapping_clean["category"] == "discharge_disposition"].copy()
source_map = mapping_clean[mapping_clean["category"] == "admission_source"].copy()

print("✅ Tablas separadas")
print("admission:", admission_map.shape)
print("discharge:", discharge_map.shape)
print("source:", source_map.shape)

✅ Tablas separadas
admission: (9, 3)
discharge: (31, 3)
source: (25, 3)


In [23]:
# 1. Enriquecer Admission Type
df_main = df_main.merge(
    admission_map.rename(columns={
        "id": "admission_type_id",
        "description": "admission_type_desc"
    }),
    on="admission_type_id",
    how="left"
)

# 2. Enriquecer Discharge
df_main = df_main.merge(
    discharge_map.rename(columns={
        "id": "discharge_disposition_id",
        "description": "discharge_desc"
    }),
    on="discharge_disposition_id",
    how="left"
)

# 3. Enriquecer Source
df_main = df_main.merge(
    source_map.rename(columns={
        "id": "admission_source_id",
        "description": "admission_source_desc"
    }),
    on="admission_source_id",
    how="left"
)

print("✅ df_main enriquecido con éxito")

✅ df_main enriquecido con éxito


## 🏁 Verificación de la Integración 

In [24]:
df_main[[
    "admission_type_id", "admission_type_desc",
    "discharge_disposition_id", "discharge_desc",
    "admission_source_id", "admission_source_desc"
]].head()

,admission_type_id,admission_type_desc,discharge_disposition_id,discharge_desc,admission_source_id,admission_source_desc
0,6,NaN,25,Not Mapped,1,Physician Referral
1,1,Emergency,1,Discharged to home,7,Emergency Room
2,1,Emergency,1,Discharged to home,7,Emergency Room
3,1,Emergency,1,Discharged to home,7,Emergency Room
4,1,Emergency,1,Discharged to home,7,Emergency Room


## 🩹 Normalización de Valores Nulos para que no quede ningun hueco estableciendo los no nulos como unknow

In [25]:
df_main["admission_type_desc"] = df_main["admission_type_desc"].fillna("Unknown")
df_main["discharge_desc"] = df_main["discharge_desc"].fillna("Unknown")
df_main["admission_source_desc"] = df_main["admission_source_desc"].fillna("Unknown")

In [26]:
# Definimos la lista de columnas a borrar
cols_to_drop = [
    # Medicamentos
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride',
    'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone',
    'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide',
    'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin',
    'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone',
    # Residuales de Merges
    'category_x', 'category_y', 'category',
    # IDs que ya fueron traducidos
    'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
    # Administrativas
    'payer_code',
]

# Ejecutamos la limpieza
df_main.drop(columns=cols_to_drop, inplace=True, errors='ignore')

print(f"✅ Limpieza completada. Ahora tu dataset tiene {df_main.shape[1]} columnas.")

✅ Limpieza completada. Ahora tu dataset tiene 26 columnas.


In [29]:
#Verificamos cuantas columnas quedaron en el dataset final
print("------- Columnas finales en el dataset principal -------")
print(df_main.columns)

------- Columnas finales en el dataset principal -------
Index(['encounter_id', 'patient_nbr', 'race', 'gender', 'time_in_hospital',
       'medical_specialty', 'num_lab_procedures', 'num_procedures',
       'num_medications', 'number_outpatient', 'number_emergency',
       'number_inpatient', 'number_diagnoses', 'change', 'diabetesMed',
       'diag_1_group', 'diag_2_group', 'diag_3_group', 'med_change_count',
       'treatment_status', 'readmitted_status', 'age_numeric',
       'hospital_stay_type', 'admission_type_desc', 'discharge_desc',
       'admission_source_desc'],
      dtype='object')


## 💾 Persistencia y Exportación del Dataset Final

In [30]:
## Exportamos el dataset limpio y enriquecido para su análisis posterior
df_main.to_csv('diabetic_data_clean.csv', index=False)